# ReefScan AI — Lightning AI backup run (E7)

Auto-detects which experiments are missing (from `runs/` manifests) and runs **the next missing one**. Re-run this notebook until it prints `ALL EXPERIMENTS COMPLETE`.

**Before Run All:** Studio machine → **GPU (T4)**.

In [ ]:
# --- 0. Clone the repo + install dependencies ------------------------------
!git clone https://github.com/Jayant1711/reefscan.git reefscan
%cd reefscan
!pip install -q -r requirements.txt


In [ ]:
# --- 1. GPU + test-suite check (fails loudly if anything is broken) --------
import torch, sys
assert torch.cuda.is_available(), "NO GPU — enable the accelerator and re-run!"
print("GPU:", torch.cuda.get_device_name(0))
!python -m pytest tests/ -q


In [ ]:
# --- 2. Auto-detect what is missing and run the next experiment ------------
import subprocess, sys
from pathlib import Path

def done(task, model, strategy):
    marker = Path("runs") / task / f"{model}_{strategy}" / "manifest.json"
    return marker.exists()

def run(cmd):
    print(">>", " ".join(cmd))
    subprocess.run([sys.executable, *cmd], check=True)

def ensure_coral(dataset="coral"):
    if not Path("/kaggle/tmp/reefscan_data/coral/imagefolders").exists() and \
       not Path("datasets/coral/imagefolders").exists():
        run(["download_data.py", "--dataset", dataset])

if not done("coral", "resnet34", "feature_extraction"):
    ensure_coral()
    run(["clean_data.py", "--dataset", "coral", "--apply"])
    run(["train.py", "--task", "coral", "--model", "all", "--strategy", "feature_extraction", "--platform", "lightning"])
    print(">>> E1 done. Re-run this notebook to continue with E2.")
elif not done("coral", "efficientnet_b0", "finetune"):
    import pandas as pd
    lb = pd.read_csv("results/coral/model_leaderboard.csv")
    top2 = lb.nlargest(2, "weighted_f1")["model"].tolist()
    for model in top2:
        if not done("coral", model, "finetune"):
            run(["train.py", "--task", "coral", "--model", model, "--strategy", "finetune", "--platform", "lightning"])
    print(">>> E2 done. Re-run this notebook to continue with E3.")
elif not done("seg", "unet_small", "from_scratch"):
    if not Path("datasets/deepfish/DeepFish").exists():
        run(["download_data.py", "--dataset", "deepfish"])
    run(["clean_data.py", "--dataset", "seg", "--apply"])
    run(["train.py", "--task", "seg", "--platform", "lightning"])
    run(["evaluate.py", "--task", "seg"])
    print(">>> E3 done. Re-run this notebook to continue with fish (E4/E5).")
elif not done("fish", "resnet34", "feature_extraction"):
    if not Path("datasets/fishnet/curated").exists() or not list(Path("datasets/fishnet/curated").rglob("*.jpg")):
        run(["download_data.py", "--dataset", "fishnet"])
        run(["download_data.py", "--prepare-fishnet"])
    run(["clean_data.py", "--dataset", "fishnet", "--apply"])
    run(["train.py", "--task", "fish", "--model", "all", "--strategy", "feature_extraction", "--platform", "lightning"])
    print(">>> E4 done. Re-run this notebook to continue with E5.")
elif not done("fish", "resnet34", "finetune"):
    import pandas as pd
    lb = pd.read_csv("results/fish/model_leaderboard.csv")
    fe = lb[lb["strategy"] == "feature_extraction"]
    winner = fe.nlargest(1, "weighted_f1")["model"].iloc[0]
    if not done("fish", winner, "finetune"):
        run(["train.py", "--task", "fish", "--model", winner, "--strategy", "finetune", "--platform", "lightning"])
    run(["evaluate.py", "--task", "fish"])
    print(">>> E5 done.")
else:
    print("ALL EXPERIMENTS COMPLETE — run the export cell below.")

In [ ]:
# --- 3. Save: ledger + artifact zip -----------------------------------------
!python scripts/log_experiment.py
!python scripts/export_artifacts.py
import glob
print("DOWNLOAD (Lightning file browser):", sorted(glob.glob("reefscan_artifacts_lightning_*.zip"))[-1])